# Grants, PII & Lineage Governance

Governance notebook for the `hop` catalog: admin users registry, grants, PII tagging, and lineage.

In [0]:
from pyspark.sql import functions as F

# ============================================================
# hop.gov.admin_users (user_name STRING)
# user_name = the account you sign in with (your email)
# ============================================================

# Current signed-in user (email)
current_user = spark.sql("SELECT current_user() AS user_name").first()["user_name"]
print("Current signed-in user:", current_user)

# Schema + table
spark.sql("CREATE SCHEMA IF NOT EXISTS hop.gov")
spark.sql("CREATE TABLE IF NOT EXISTS hop.gov.admin_users (user_name STRING)")

# Add the current user once (skip if already present)
spark.sql(f"""
    INSERT INTO hop.gov.admin_users
    SELECT '{current_user}'
    WHERE NOT EXISTS (
        SELECT 1 FROM hop.gov.admin_users
        WHERE user_name = '{current_user}'
    )
""")

display(spark.table("hop.gov.admin_users"))

In [0]:
from pyspark.sql import functions as F

# ============================================================
# STEP 1: silver.customers se masking hata do
# (mask reference dropped function ko point karta hai, isliye
# table invalid hai. Pehle function wapas bana kar table valid
# karo, phir masks DROP MASK karo, phir function drop)
# ============================================================

# 1a. Function temporarily recreate (dependency loop todkar)
spark.sql("""
    CREATE OR REPLACE FUNCTION hop.gov.mask_contact(v STRING)
    RETURNS STRING
    RETURN v
""")

# 1b. Dono columns se mask hatao
spark.sql("ALTER TABLE hop.silver.customers ALTER COLUMN email DROP MASK")
spark.sql("ALTER TABLE hop.silver.customers ALTER COLUMN phone DROP MASK")
print("silver.customers masks removed — table readable again")

# 1c. Function ab chahiye nahi
spark.sql("DROP FUNCTION IF EXISTS hop.gov.mask_contact")

# ============================================================
# STEP 2: hop.raw schema + silver.customers ka pura data
# ============================================================

spark.sql("CREATE SCHEMA IF NOT EXISTS hop.raw")
spark.sql("CREATE OR REPLACE TABLE hop.raw.customers AS SELECT * FROM hop.silver.customers")

print("silver.customers rows:", spark.table("hop.silver.customers").count())
print("raw.customers rows:", spark.table("hop.raw.customers").count())

display(spark.table("hop.raw.customers").limit(5))

In [0]:
from pyspark.sql import functions as F

# ============================================================
# hop.gov.mask_contact(v STRING)
# - hop.gov.admin_users walo ko full value dikhta hai
# - baaki sabko '****' + last 4 characters
# ============================================================

spark.sql("""
    CREATE OR REPLACE FUNCTION hop.gov.mask_contact(v STRING)
    RETURNS STRING
    COMMENT 'Full contact value for hop.gov.admin_users; **** + last 4 chars for everyone else'
    RETURN
        CASE
            WHEN current_user() IN (SELECT user_name FROM hop.gov.admin_users) THEN v
            WHEN v IS NULL THEN NULL
            ELSE CONCAT('****', RIGHT(v, 4))
        END
""")

# ============================================================
# Column mask apply karo hop.raw.customers par
# ============================================================

spark.sql("ALTER TABLE hop.raw.customers ALTER COLUMN email SET MASK hop.gov.mask_contact")
spark.sql("ALTER TABLE hop.raw.customers ALTER COLUMN phone SET MASK hop.gov.mask_contact")

# ============================================================
# Verify
# ============================================================

print("Masks registered on hop.raw.customers:")
display(spark.sql("DESCRIBE TABLE EXTENDED hop.raw.customers").filter(F.col("col_name").isin("email", "phone")))

print("Current user:", spark.sql("SELECT current_user() AS u").first()["u"], "(admin_users -> full values):")
display(spark.table("hop.raw.customers").select("customer_id", "email", "phone").limit(5))

print("Non-admin users (demo):")
display(
    spark.table("hop.raw.customers")
    .select("customer_id", F.expr("CONCAT('****', RIGHT(email, 4))").alias("email_masked"), F.expr("CONCAT('****', RIGHT(phone, 4))").alias("phone_masked"))
    .limit(5)
)

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Proof tables: hop.raw.customers ka data
# - proof_customers_admin  -> admin jaisa dekhta hai (full)
# - proof_customers_masked -> non-admin jaisa dekhta hai
#                          ('****' + last 4 chars, mask_contact
#                          ke ELSE branch wahi logic)
# ============================================================

# --- Admin view (current user admin_users mein hai -> full values)
admin_df = spark.table("hop.raw.customers").select("customer_id", "email", "phone")
admin_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("hop.gov.proof_customers_admin")

# --- Non-admin view (masked proof)
masked_df = (
    spark.table("hop.raw.customers")
    .select(
        "customer_id",
        F.expr("CONCAT('****', RIGHT(email, 4))").alias("email"),
        F.expr("CONCAT('****', RIGHT(phone, 4))").alias("phone")
    )
)
masked_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("hop.gov.proof_customers_masked")

# --- Verify
print("proof_customers_admin rows:", spark.table("hop.gov.proof_customers_admin").count())
print("proof_customers_masked rows:", spark.table("hop.gov.proof_customers_masked").count())

print("Admin view (full values):")
display(spark.table("hop.gov.proof_customers_admin").limit(5))

print("Non-admin view (masked values):")
display(spark.table("hop.gov.proof_customers_masked").limit(5))

In [0]:
from pyspark.sql import functions as F

# ============================================================
# Grants to `account users`:
# - USE CATALOG on hop
# - USE SCHEMA + SELECT on hop.gold aur hop.analytics
# - USE SCHEMA only on hop.raw (koi SELECT nahi)
# - SELECT kabhi catalog par nahi, na hi hop.raw / hop.raw.customers par
# ============================================================

# hop.analytics schema (agar exist nahi karta)
spark.sql("CREATE SCHEMA IF NOT EXISTS hop.analytics")

# --- USE CATALOG (sirf schema use karne ke liye, koi SELECT nahi)
spark.sql("GRANT USE CATALOG ON CATALOG hop TO `account users`")

# --- hop.gold: USE SCHEMA + SELECT (schema par, catalog par nahi)
spark.sql("GRANT USE SCHEMA ON SCHEMA hop.gold TO `account users`")
spark.sql("GRANT SELECT ON SCHEMA hop.gold TO `account users`")

# --- hop.analytics: USE SCHEMA + SELECT
spark.sql("GRANT USE SCHEMA ON SCHEMA hop.analytics TO `account users`")
spark.sql("GRANT SELECT ON SCHEMA hop.analytics TO `account users`")

# --- hop.raw: sirf USE SCHEMA (masked data read allowed nahi)
spark.sql("GRANT USE SCHEMA ON SCHEMA hop.raw TO `account users`")

# --- Pakka karo: kahin SELECT on catalog / raw na rahe
spark.sql("REVOKE SELECT ON CATALOG hop FROM `account users`")
spark.sql("REVOKE SELECT ON SCHEMA hop.raw FROM `account users`")
spark.sql("REVOKE SELECT ON TABLE hop.raw.customers FROM `account users`")

# ============================================================
# Verify
# ============================================================

print("Grants on CATALOG hop:")
display(spark.sql("SHOW GRANTS ON CATALOG hop"))

print("Grants on SCHEMA hop.gold:")
display(spark.sql("SHOW GRANTS ON SCHEMA hop.gold"))

print("Grants on SCHEMA hop.analytics:")
display(spark.sql("SHOW GRANTS ON SCHEMA hop.analytics"))

print("Grants on SCHEMA hop.raw:")
display(spark.sql("SHOW GRANTS ON SCHEMA hop.raw"))